# Semana 11: Determinantes, Inversas y Rango
## Notebook Conceptual (NB1) - Explorando Matrices con Datos Dummy

### Propósito de la sesión
Comprender la **capacidad** de una matriz y cómo la información fluye a través de ella. Exploraremos el **determinante** (factor de escala de volumen), la **inversa** (que "deshace" la transformación) y el **rango** (dimensión efectiva de la transformación). Conectaremos estos conceptos con aplicaciones en machine learning (distribución normal multivariante), deep learning (capacidad expresiva de capas) y robótica (Jacobiano).

### Objetivos de aprendizaje
*   Calcular el **determinante** de una matriz y relacionarlo con el cambio de área/volumen.
*   Calcular la **inversa** de una matriz y usarla para resolver sistemas lineales.
*   Entender el **rango** de una matriz como medida de su capacidad.
*   Relacionar determinante, inversa y rango con la **invertibilidad**.
*   Conectar estos conceptos con aplicaciones en **ML** (normal multivariante), **DL** (rango de pesos) y **robótica** (determinante del Jacobiano).

## Configuración Inicial

Importamos las librerías necesarias: NumPy para operaciones, Matplotlib para visualizaciones.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Determinante: Factor de Escala de Área/Volumen

El determinante de una matriz cuadrada $\mathbf{A} \in \mathbb{R}^{n \times n}$ mide cuánto se escala el área (en 2D) o volumen (en 3D) al aplicar la transformación lineal definida por $\mathbf{A}$.

### 1.1. Interpretación geométrica en 2D

In [ ]:
# Definimos un cuadrado unitario
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1]])

def plot_transform(A, title):
    """Aplica la transformación A al cuadrado unitario y visualiza."""
    transformed = square @ A.T
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Original
    ax1.add_patch(Polygon(square, closed=True, alpha=0.3, color='blue'))
    ax1.scatter(square[:, 0], square[:, 1], color='red', s=50)
    ax1.set_xlim(-2, 3)
    ax1.set_ylim(-2, 3)
    ax1.set_aspect('equal')
    ax1.grid(True)
    ax1.set_title('Cuadrado unitario original')
    
    # Transformado
    ax2.add_patch(Polygon(transformed, closed=True, alpha=0.3, color='green'))
    ax2.scatter(transformed[:, 0], transformed[:, 1], color='red', s=50)
    ax2.set_xlim(-3, 4)
    ax2.set_ylim(-3, 4)
    ax2.set_aspect('equal')
    ax2.grid(True)
    ax2.set_title(f'Transformado: det(A) = {np.linalg.det(A):.2f}')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Matriz con determinante > 0 (rotación + escalado)
A1 = np.array([[2, 0.5],
               [0.3, 1.5]])
plot_transform(A1, 'Transformación con det(A) > 0')

# Matriz con determinante negativo (refleja + escala)
A2 = np.array([[1.5, 1],
               [1, -1]])
plot_transform(A2, 'Transformación con det(A) < 0')

# Matriz con determinante cercano a cero (aplasta el área)
A3 = np.array([[1, 0.9],
               [1, 0.9]])
plot_transform(A3, 'Transformación con det(A) ≈ 0')

### 1.2. Cálculo de determinantes con NumPy

In [ ]:
# Matriz 2x2
A2x2 = np.array([[3, 4],
                 [2, 1]])
det_2x2 = np.linalg.det(A2x2)
print(f"Matriz A2x2:\n{A2x2}")
print(f"Determinante: {det_2x2:.2f}")
print(f"Verificación manual: (3*1 - 4*2) = {3*1 - 4*2}\n")

# Matriz 3x3
A3x3 = np.array([[2, -1, 0],
                 [3, 2, 1],
                 [1, 2, 3]])
det_3x3 = np.linalg.det(A3x3)
print(f"Matriz A3x3:\n{A3x3}")
print(f"Determinante: {det_3x3:.2f}")

# Matriz singular (determinante cero)
A_sing = np.array([[1, 2],
                   [2, 4]])
det_sing = np.linalg.det(A_sing)
print(f"\nMatriz singular:\n{A_sing}")
print(f"Determinante: {det_sing:.2f}")

---
## 2. Matriz Inversa: "Deshaciendo" la Transformación

Una matriz $\mathbf{A}$ es invertible si existe $\mathbf{A}^{-1}$ tal que $\mathbf{A} \mathbf{A}^{-1} = \mathbf{A}^{-1} \mathbf{A} = \mathbf{I}$. Esto es posible si y solo si $\det(\mathbf{A}) \neq 0$.

In [ ]:
# Matriz invertible
A_inv_test = np.array([[4, 7],
                       [2, 6]])

# Calculamos la inversa
A_inv = np.linalg.inv(A_inv_test)
print(f"Matriz A:\n{A_inv_test}")
print(f"\nInversa A⁻¹:\n{A_inv}")

# Verificamos que A @ A⁻¹ = I
identidad = A_inv_test @ A_inv
print(f"\nA @ A⁻¹:\n{identidad}")

# Resolvemos un sistema Ax = b
b = np.array([1, 2])
x_inv = A_inv @ b
x_solve = np.linalg.solve(A_inv_test, b)
print(f"\nSolución con inversa: {x_inv}")
print(f"Solución con solve: {x_solve}")

In [ ]:
# Intento de invertir una matriz singular
A_sing_inv = np.array([[1, 2],
                       [2, 4]])

try:
    A_sing_inv_inv = np.linalg.inv(A_sing_inv)
    print("Inversa calculada:", A_sing_inv_inv)
except np.linalg.LinAlgError as e:
    print(f"Error al invertir: {e}")
    print("Una matriz singular (det=0) no tiene inversa.")

---
## 3. Rango de una Matriz: Capacidad Expresiva

El **rango** de una matriz $\mathbf{A} \in \mathbb{R}^{m \times n}$ es el número máximo de columnas (o filas) linealmente independientes. Indica la dimensión de la imagen de la transformación.

*   Si $\mathbf{A}$ es cuadrada y tiene rango completo ($= n$), entonces es invertible y $\det(\mathbf{A}) \neq 0$.
*   Si el rango es menor que $n$, la matriz es singular (no invertible).

In [ ]:
# Matriz de rango completo (3x3)
A_rank_full = np.array([[1, 2, 3],
                        [4, 5, 6],
                        [7, 8, 9]])
# Nota: esta matriz tiene determinante cero (filas linealmente dependientes?)
rank_full = np.linalg.matrix_rank(A_rank_full)
det_full = np.linalg.det(A_rank_full)
print(f"Matriz A (3x3):\n{A_rank_full}")
print(f"Rango: {rank_full} (debería ser 2?)")
print(f"Determinante: {det_full:.2f}")
print("📌 En realidad, las filas son (1,2,3), (4,5,6), (7,8,9) que son linealmente dependientes (fila3 = 2*fila2 - fila1).")

# Matriz de rango completo (aleatoria)
A_random = np.random.randn(4, 4)
rank_random = np.linalg.matrix_rank(A_random)
det_random = np.linalg.det(A_random)
print(f"\nMatriz aleatoria 4x4:")
print(f"Rango: {rank_random} (máximo 4)")
print(f"Determinante: {det_random:.4f}")

# Matriz con rango deficiente (columna dependiente)
A_rank_def = np.array([[1, 2, 3],
                       [2, 4, 6],
                       [3, 6, 9]])  # todas las columnas son múltiplos
rank_def = np.linalg.matrix_rank(A_rank_def)
det_def = np.linalg.det(A_rank_def)
print(f"\nMatriz con rango deficiente:\n{A_rank_def}")
print(f"Rango: {rank_def}")
print(f"Determinante: {det_def:.2f}")

### 3.1. Relación entre determinante, inversa y rango

Para una matriz cuadrada $\mathbf{A} \in \mathbb{R}^{n \times n}$:

| Rango | Determinante | Invertible? |
|-------|--------------|-------------|
| $n$ (completo) | $\neq 0$ | Sí |
| $< n$ | $0$ | No |

In [ ]:
# Demostración con una matriz 3x3 de rango 2
A_demo = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])

rank_demo = np.linalg.matrix_rank(A_demo)
det_demo = np.linalg.det(A_demo)

print(f"Matriz A:\n{A_demo}")
print(f"Rango: {rank_demo}")
print(f"Determinante: {det_demo:.2e}")
try:
    inv_demo = np.linalg.inv(A_demo)
    print("Inversa calculada (no debería)")
except np.linalg.LinAlgError:
    print("La matriz no es invertible (como esperábamos).")

---
## 4. Conexiones IA

### 4.1. ML: Distribución Normal Multivariante

La función de densidad de la normal multivariante incluye el determinante de la matriz de covarianza:

$$\mathcal{N}(\mathbf{x}|\boldsymbol{\mu}, \boldsymbol{\Sigma}) = \frac{1}{(2\pi)^{d/2}|\boldsymbol{\Sigma}|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x} - \boldsymbol{\mu})\right)$$

In [ ]:
# Ejemplo: matriz de covarianza y su determinante
Sigma = np.array([[2.0, 0.8],
                  [0.8, 1.5]])
det_Sigma = np.linalg.det(Sigma)
print(f"Matriz de covarianza Σ:\n{Sigma}")
print(f"Determinante |Σ| = {det_Sigma:.4f}")
print("Este determinante aparece en el factor de normalización.")

### 4.2. DL: Rango de Matrices de Pesos

El rango de la matriz de pesos de una capa fully connected indica su **capacidad expresiva**. Si el rango es bajo, la capa proyecta las entradas en un subespacio de menor dimensión, perdiendo información.

In [ ]:
# Simulamos una capa con 10 entradas y 5 salidas
W = np.random.randn(5, 10)
rank_W = np.linalg.matrix_rank(W)
print(f"Matriz de pesos W (5x10). Rango: {rank_W} (máximo posible 5)")

# Introducimos redundancia: hacemos que una columna sea combinación lineal
W[:, 3] = 0.5 * W[:, 1] + 0.7 * W[:, 5]
rank_W_new = np.linalg.matrix_rank(W)
print(f"Después de introducir dependencia lineal, rango: {rank_W_new}")

print("\n📌 En redes profundas, se busca mantener rangos altos para preservar capacidad.")

### 4.3. Robótica: Determinante del Jacobiano

En robótica, el Jacobiano $\mathbf{J}(\mathbf{q})$ relaciona velocidades articulares $\dot{\mathbf{q}}$ con la velocidad del efector final $\mathbf{v}$:

$$\mathbf{v} = \mathbf{J}(\mathbf{q}) \dot{\mathbf{q}}$$

El determinante del Jacobiano (para un manipulador no redundante) indica si la configuración es singular. Si $\det(\mathbf{J}) = 0$, hay direcciones inalcanzables.

In [ ]:
# Simulamos un Jacobiano 2x2 (2 articulaciones, 2 dimensiones)
J_sing = np.array([[1, 2],
                   [2, 4]])  # singular
J_nonsing = np.array([[2, 1],
                      [1, 2]])

det_J_sing = np.linalg.det(J_sing)
det_J_nonsing = np.linalg.det(J_nonsing)

print(f"Jacobiano singular:\n{J_sing}\ndet = {det_J_sing:.2f}")
print(f"\nJacobiano no singular:\n{J_nonsing}\ndet = {det_J_nonsing:.2f}")

print("\n📌 Cerca de singularidades, el determinante se hace pequeño, indicando pérdida de grados de libertad.")

---
## Ejercicios para el Estudiante

1.  **Determinante de matrices aleatorias:** Genera 10 matrices aleatorias 5x5 y calcula sus determinantes. ¿Cuántas son singulares? ¿Por qué es casi imposible obtener exactamente un determinante cero?

2.  **Rango y ruido:** Toma una matriz de rango deficiente (ej. la matriz con filas dependientes) y añádele un pequeño ruido (ej. `A + 1e-5 * np.random.randn(*A.shape)`). Calcula su rango y determinante. ¿Qué observas?

3.  **Inversa vs. solve:** Para una matriz 100x100 bien condicionada, compara el tiempo de `np.linalg.inv(A) @ b` vs `np.linalg.solve(A, b)`. ¿Cuál es más rápido? ¿Por qué?

4.  **Interpretación del determinante:** Crea una matriz 2x2 con determinante 2. Aplica esta transformación a un círculo de radio 1 (puedes generar puntos en el círculo). ¿Qué figura obtienes? ¿Cómo se relaciona el área con el determinante?

5.  **Reflexión:** En una red neuronal, ¿qué implicaciones tiene que la matriz de pesos de una capa tenga rango bajo? ¿Cómo podría afectar esto al entrenamiento y a la capacidad de generalización?

---
## Conclusiones de la Sesión

*   El **determinante** mide el cambio de volumen que induce una transformación lineal. Es cero si la matriz es singular (aplasta el espacio).
*   La **inversa** permite "deshacer" una transformación lineal, pero solo existe si el determinante es no nulo (matriz invertible).
*   El **rango** indica la dimensión de la imagen de la transformación. Un rango completo (para matrices cuadradas) equivale a invertibilidad.
*   Estos conceptos están íntimamente relacionados: $\det(\mathbf{A}) \neq 0$ ⇔ $\mathbf{A}$ invertible ⇔ $\text{rango}(\mathbf{A}) = n$.
*   En **machine learning**, el determinante aparece en la normal multivariante (matriz de covarianza).
*   En **deep learning**, el rango de las matrices de pesos indica la capacidad expresiva de las capas.
*   En **robótica**, el determinante del Jacobiano detecta singularidades.

¡Ahora comprendes cómo estas propiedades matriciales determinan el flujo de información en los modelos de IA!